In [1]:
import duckdb
from co2sat.utils import data_dir

In [2]:
con = duckdb.connect()

In [3]:
p = str(data_dir("processed", "dynamic_features.parquet"))

In [4]:
# Hours present per (facility, date)
completeness = con.execute(f"""
    SELECT facility_id, date, COUNT(*) AS n_hours
    FROM read_parquet('{p}')
    GROUP BY facility_id, date
""").df()

In [5]:
print(completeness["n_hours"].value_counts().sort_index(ascending=False))

n_hours
24    155172
22      1158
14      1158
Name: count, dtype: int64


In [6]:
affected = completeness[completeness["n_hours"] < 24]
print(affected.groupby("date")["n_hours"].first())
print(f"\nPlant-days with full 24h: {(completeness['n_hours'] == 24).mean():.1%}")

date
2021-04-29    22
2022-09-13    14
Name: n_hours, dtype: int64

Plant-days with full 24h: 98.5%


In [ ]:
nan_stats = (
    con.execute(f"""
    SELECT {
        ", ".join(
            f"SUM(CASE WHEN band_{b:02d} IS NULL OR isnan(band_{b:02d}) THEN 1 ELSE 0 END) AS nan_b{b:02d}"
            for b in range(1, 17)
        )
    }
    FROM read_parquet('{p}')
""")
    .df()
    .T
)
print(nan_stats)

               0
nan_b01  16615.0
nan_b02  18365.0
nan_b03  17543.0
nan_b04  18538.0
nan_b05  13628.0
nan_b06  11670.0
nan_b07  15696.0
nan_b08  16986.0
nan_b09  18149.0
nan_b10  16596.0
nan_b11  13751.0
nan_b12  12520.0
nan_b13  18218.0
nan_b14  17270.0
nan_b15  14374.0
nan_b16  15775.0


In [8]:
epa = str(data_dir("processed", "epa_daily_with_attributes.parquet"))
coverage = con.execute(f"""
    WITH labels AS (
        SELECT DISTINCT facility_id, CAST(date AS DATE) AS date
        FROM read_parquet('{epa}')
        WHERE co2_metric_tons > 0 AND gross_load_mwh > 0
    ),
    sat AS (
        SELECT facility_id, date, COUNT(*) AS n_hours
        FROM read_parquet('{p}')
        GROUP BY facility_id, date
    )
    SELECT
        COUNT(*) AS labeled_plant_days,
        SUM(CASE WHEN s.n_hours IS NOT NULL THEN 1 ELSE 0 END) AS with_satellite,
        SUM(CASE WHEN s.n_hours = 24 THEN 1 ELSE 0 END) AS with_full_24h
    FROM labels l LEFT JOIN sat s USING (facility_id, date)
""").df()
print(coverage)

   labeled_plant_days  with_satellite  with_full_24h
0              507574         90261.0        88795.0
